# Gold — dim_google_review_scd1 (SCD 1, poc method)

One row per Google review (Place Details, accumulated).
Method (poc `scd_type_1`, helper `_scd`): **changed keys** in the window → re-read their full current rows → metadata `etl_date` / `etl_by` → `add_checksums` → **full**: overwrite / **incremental**: MERGE on `primary_checksum`, update only when `record_checksum` differs → Unknown row → grain check. Widgets: `load_type`, `from_date`, `to_date` (Vietnam days, empty = yesterday .. tomorrow). Runs in task `gold_dims_scd1` (nothing depends on it); the live dim is unchanged.

**Changed keys** = reviews read in the window (`stg_restaurant_place_reviews.last_seen_at`) **∪** reviews already in the table with `restaurant_id = 'unknown'` (the place may have matched a restaurant since). `restaurant_id` = current `dim_restaurant` row of the place. `created_at` = `first_seen_at`, `modified_at` = `last_seen_at` (not hashed). Skipped when the Silver table does not exist yet.

In [ ]:
%run ./_scd

In [ ]:
for layer in ("silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full"])
dbutils.widgets.text("from_date", "")   # yyyy-MM-dd, Vietnam day, inclusive; empty = yesterday
dbutils.widgets.text("to_date", "")     # exclusive; empty = tomorrow (today's reads included)
SILVER, GOLD = dbutils.widgets.get("silver"), dbutils.widgets.get("gold")
LOAD_TYPE = dbutils.widgets.get("load_type")
FROM_DATE, TO_DATE = default_dates(dbutils.widgets.get("from_date"), dbutils.widgets.get("to_date"))
FULL = LOAD_TYPE == "full"
print("full: every key" if FULL else f"incremental: keys changed in [{FROM_DATE}, {TO_DATE}) (Vietnam days)")

In [ ]:
TABLE = f"{GOLD}.dim_google_review_scd1"
REVIEWS = f"{SILVER}.stg_restaurant_place_reviews"
if not spark.catalog.tableExists(REVIEWS):
    dbutils.notebook.exit(f"{REVIEWS} does not exist yet: {TABLE} not built")

reviews = spark.table(REVIEWS).where("review_id IS NOT NULL")
if not FULL:
    frames = [reviews.where(in_window(vn_date("last_seen_at"), FROM_DATE, TO_DATE)).select("review_id")]
    if spark.catalog.tableExists(TABLE):
        frames.append(spark.table(TABLE).where("review_key <> -1 AND restaurant_id = 'unknown'").select("review_id"))
    reviews = reviews.join(changed_keys(*frames), "review_id", "left_semi")

places = (spark.table(f"{GOLD}.dim_restaurant")
          .where("is_current AND restaurant_id <> 'unknown' AND place_id IS NOT NULL")
          .groupBy("place_id").agg(F.min("restaurant_id").alias("restaurant_id")))
src = (reviews.join(places, "place_id", "left")
       .select(F.xxhash64("review_id").alias("review_key"), "review_id",
               F.coalesce("restaurant_id", F.lit("unknown")).alias("restaurant_id"), "place_id",
               F.coalesce(F.date_format(F.from_utc_timestamp("publish_time", TZ), "yyyyMMdd").cast("int"), F.lit(-1)).alias("publish_date_key"),
               F.col("rating").cast("tinyint").alias("rating"), "review_rank", "review_text", "review_language",
               "original_text", "original_language", "author_name", "publish_time", "relative_time", "google_maps_uri",
               F.col("first_seen_at").alias("created_at"), F.col("last_seen_at").alias("modified_at"))
       .withColumn("etl_date", F.current_timestamp())
       .withColumn("etl_by", F.lit("Databricks")))
src = add_checksums(src, ["review_id"], exclude=["review_key", "created_at", "modified_at"])

In [ ]:
load_scd1(src, TABLE, """
        review_key BIGINT, review_id STRING, restaurant_id STRING, place_id STRING, publish_date_key INT, rating TINYINT,
        review_rank INT, review_text STRING, review_language STRING, original_text STRING, original_language STRING,
        author_name STRING, publish_time TIMESTAMP, relative_time STRING, google_maps_uri STRING,
        created_at TIMESTAMP, modified_at TIMESTAMP, etl_date TIMESTAMP, etl_by STRING,
        primary_checksum STRING, record_checksum STRING""",
    "Google review dimension, SCD 1 (poc checksum method). review_key = xxhash64(review_id), -1 = Unknown.",
    ["review_id"], LOAD_TYPE, {"review_key": "-1", "restaurant_id": "'unknown'", "publish_date_key": "-1",
                               "review_text": "'Unknown'", "etl_date": "current_timestamp()", "etl_by": "'Databricks'"})